# Загрузка и установка необходимых инструментов

In [ ]:
!pip install -r ultralytics opencv-python-headless numpy pandas boto3 matplotlib seaborn

from pathlib import Path
import os
from ultralytics import YOLO
import shutil
import pandas as pd

# Загрузка данных

In [ ]:
BASE_DIR = Path(os.getcwd())

DATASET_ROOT = BASE_DIR / "data" / "dataset"


if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Датасет не найден по пути: {DATASET_ROOT}. Проверьте структуру папок!")

train_images = DATASET_ROOT / "train" / "images"
test_images  = DATASET_ROOT / "test"  / "images"

print(f"Ищем тренировочные данные в: {train_images}")
print(f"Ищем тестовые данные в: {test_images}")

exps = [d.name for d in train_images.iterdir() if d.is_dir()]

if not exps:
    raise ValueError(f"Не найдено папок с экспериментами в {train_images}. Проверьте структуру архива!")

print(f"Найдено экспериментов: {exps}")

exp_to_images = {}
for exp in exps:
    exp_dir = train_images / exp
    images = list(exp_dir.rglob("*.tif"))
    if images:
        exp_to_images[exp] = images
    else:
        print(f"Предупреждение: В папке {exp} не найдено .tif файлов")

# Тестовый набор
test_exp_name = "19112020 FAD control" 
test_exp_dir = test_images / test_exp_name

if test_exp_dir.exists():
    test_images_list = list(test_exp_dir.rglob("*.tif"))
    print(f"Найдено тестовых изображений: {len(test_images_list)}")
else:
    print(f"ОШИБКА: Папка тестового набора не найдена: {test_exp_dir}")
    test_images_list = []


CV_DIR = Path("./cv_splits")
CV_DIR.mkdir(exist_ok=True)

test_list_path = CV_DIR / "test.txt"

with open(test_list_path, "w") as f:
    for img in test_images_list:        
        f.write(str(img.resolve()) + "\n")

print(f"Тестовый список сохранен в: {test_list_path}")

# Создание yaml- файлов

In [ ]:
for val_exp in exps:
    train_images_paths = []
    for exp in exps:
        if exp == val_exp:
            continue
        train_images_paths.extend(exp_to_images[exp])
    
    val_images_paths = exp_to_images[val_exp]
    
    train_list_file = CV_DIR / f"train_fold_{val_exp}.txt"
    val_list_file   = CV_DIR / f"val_fold_{val_exp}.txt"
    
    with open(train_list_file, "w") as f:
        for p in train_images_paths:
            f.write(str(p.resolve()) + "\n")

    with open(val_list_file, "w") as f:
        for p in val_images_paths:
            f.write(str(p.resolve()) + "\n")
    
    yaml_content = f"""
path: {DATASET_ROOT.resolve()}
train: {train_list_file.resolve()}
val: {val_list_file.resolve()}
test: {test_list_path.resolve()}

nc: 1
names: ['astrocyte']
channels: 1
"""
    yaml_content = "\n".join(line.strip() for line in yaml_content.splitlines())

    yaml_path = CV_DIR / f"fold_{val_exp}.yaml"
    yaml_path.write_text(yaml_content)
    
    print(f"Создан YAML для фолда {val_exp}: {yaml_path}")

# Загрузка предобученной модели

In [ ]:
model = YOLO("yolo11m-seg.pt")

# Запуск обучения с аугментациями, кросс-валидацией. Если нужно, можно заморорзить первые слои 

In [ ]:
MODELS_DIR = Path("./saved_models")
MODELS_DIR.mkdir(exist_ok=True)

all_metrics = []

print(f"Начало Cross-Validation. Всего фолдов: {len(exps)}")

for val_exp in exps:
    model = YOLO("yolo11m-seg.pt")
    
    #Заморозка первых 9 модулей (backbone), если нужна
    
    #modules = list(model.model.named_modules())
    #for i, (name, module) in enumerate(modules):
    #    for param in module.parameters():
    #        param.requires_grad = (i >= 151)
    
    yaml_path = CV_DIR / f"fold_{val_exp}.yaml"
    
    if not yaml_path.exists():
        print(f"Пропуск {val_exp}: файл {yaml_path} не найден")
        continue
    
    print(f"\n===== ОБУЧЕНИЕ ФОЛДА: {val_exp} =====")
    
    try:
        results = model.train(
            data=str(yaml_path),
            epochs=100,
            imgsz=512,
            hsv_v=0.4,
            degrees=20.0,
            translate=0.1,
            scale=0.2,
            flipud=0.1,
            fliplr=0.5,
            batch=32,
            project="./runs/segment", 
            name=f"fold_{val_exp}",  
            cache=True,              
            workers=8,               
            verbose=True,
            exist_ok=True,           
            patience=50,
            lr0=0.0001,        
            lrf=0.01, 
            optimizer='AdamW', 
            weight_decay=0.0005
        )
        
        # Сбор метрик 
        r_dict = results.results_dict
        
        metrics = {
            'fold': val_exp,
            'precision': r_dict.get('metrics/precision(B)', 0),
            'recall': r_dict.get('metrics/recall(B)', 0),
            'map50': r_dict.get('metrics/mAP50(B)', 0),
            'map50_95': r_dict.get('metrics/mAP50-95(B)', 0)
        }
        all_metrics.append(metrics)
        
        
        save_dir = Path(results.save_dir) # Это надежный способ получить путь к папке результатов
        weights_dir = save_dir / "weights"
        
        best_pt_source = weights_dir / "best.pt"
        dest_model_name_best = f"model_fold_{val_exp}_best.pt"
        dest_model_path_best = MODELS_DIR / dest_model_name_best
        
        if best_pt_source.exists():
            shutil.copy2(best_pt_source, dest_model_path_best)
            print(f"Лучшая модель сохранена: {dest_model_path_best}")
        else:
            print(f"Файл best.pt не найден по пути {best_pt_source}")
        
        last_pt_source = weights_dir / "last.pt"
        dest_model_name_last = f"model_fold_{val_exp}_last.pt"
        dest_model_path_last = MODELS_DIR / dest_model_name_last
        
        if last_pt_source.exists():
            shutil.copy2(last_pt_source, dest_model_path_last)
            print(f"Последняя модель сохранена: {dest_model_path_last}")
        else:
            print(f"Файл last.pt не найден по пути {last_pt_source}")
        
        if not best_pt_source.exists() or not last_pt_source.exists():
            print(f"Содержимое папки весов: {list(weights_dir.glob('*'))}")

    except Exception as e:
        print(f"Ошибка при обучении фолда {val_exp}: {e}")
        all_metrics.append({'fold': val_exp, 'error': str(e)})

# Вывод и сохранение метрик обучения

In [ ]:
if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    
    # Сохраняем таблицу метрик в CSV
    metrics_csv_path = MODELS_DIR / "cv_metrics_summary.csv"
    df_metrics.to_csv(metrics_csv_path, index=False)
    
    print("\n" + "="*30)
    print("ИТОГОВЫЕ МЕТРИКИ CROSS-VALIDATION")
    print("="*30)
    print(df_metrics.to_string(index=False))
    
    # Вычисляем среднее и стандартное отклонение для ключевых метрик
    numeric_cols = ['precision', 'recall', 'map50', 'map50_95']
    # Фильтруем только числовые колонки, если есть ошибки (строки)
    numeric_df = df_metrics[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    if not numeric_df.empty:
        print("\n--- Средние значения и разброс ---")
        for col in numeric_cols:
            mean_val = numeric_df[col].mean()
            std_val = numeric_df[col].std()
            print(f"{col:12s}: {mean_val:.4f} +/- {std_val:.4f}")
            
    print(f"\nТаблица метрик сохранена в: {metrics_csv_path}")
    print(f"Все модели сохранены в папку: {MODELS_DIR.absolute()}")
else:
    print("Метрики не были собраны")

# Тестирование моделей

In [ ]:
BASE_DIR = Path(os.getcwd())
MODELS_DIR = BASE_DIR / "models"                 # папка с .pt файлами
DATASET_ROOT = BASE_DIR / "data" / "dataset"  # корень датасета

# Используем всю папку images (со всеми подпапками)
TEST_IMG_REL = "images/19112020 FAD control"

# Параметры тестирования
CONF_THRESHOLD = 0.25
IMGSZ = 512
BATCH = 1
IOU = 0.5


In [ ]:
# Создание YAML
test_yaml = BASE_DIR / "temp_test_dataset.yaml"
data = {
    'path': str(DATASET_ROOT / "test"),   # .../checked_dataset/test
    'test': TEST_IMG_REL,                 # "images"
    'train': TEST_IMG_REL,                # обязательное поле
    'val': TEST_IMG_REL, 
    'nc': 1,
    'names': ['astrocyte'],
    'channels': 1
}
with open(test_yaml, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)
print(f"YAML создан: {test_yaml}")
print(f"   path = {data['path']}")
print(f"   test = {data['test']}")

img_root = Path(data['path']) / data['test']
if img_root.exists():
    img_files = list(img_root.rglob("*.tif"))
    print(f"Найдено изображений: {len(img_files)}")
else:
    print(f"Папка {img_root} не найдена")


model_paths = sorted(MODELS_DIR.glob("*.pt"))
print(f"\nНайдено моделей: {len(model_paths)}")
for mp in model_paths:
    print(f"   {mp.name}")

if not model_paths:
    raise FileNotFoundError("Нет .pt файлов в папке 'models'")



In [ ]:
results = []
for model_path in model_paths:
    print(f"\nТестируем: {model_path.name}")
    try:
        model = YOLO(str(model_path))
        metrics = model.val(
            data=str(test_yaml),
            split='test',
            conf=CONF_THRESHOLD,
            iou=IOU,
            imgsz=IMGSZ,
            batch=BATCH,
            verbose=False
        )
        seg = metrics.seg
        row = {
            'model': model_path.name,
            'Precision': seg.p,
            'Recall': seg.r,
            'mAP50': seg.map50,
            'mAP50-95': seg.map
        }
        results.append(row)
        print(f"   Precision={seg.p:.4f}, Recall={seg.r:.4f}, mAP50={seg.map50:.4f}, mAP50-95={seg.map:.4f}")
    except Exception as e:
        print(f"   Ошибка: {e}")
        results.append({
            'model': model_path.name,
            'Precision': None,
            'Recall': None,
            'mAP50': None,
            'mAP50-95': None
        })



In [ ]:
df = pd.DataFrame(results)
output_csv = BASE_DIR / "models_test_results_all.csv"
df.to_csv(output_csv, index=False)
print(f"\nРезультаты сохранены в: {output_csv}")
print(df.to_string(index=False))
